In [2]:
import random
import csv
from pathlib import Path
import pandas as pd

heerlen_addresses_df = pd.read_csv("../output/heerlen_addresses.csv")

# Use prevalidated addresses and coordinates, same pattern as clients generator.
required_columns = {"full_address", "coordinates"}
missing_columns = required_columns - set(heerlen_addresses_df.columns)
if missing_columns:
    raise KeyError(f"Missing required columns in heerlen_addresses_df: {sorted(missing_columns)}")

VALID_HEERLEN_ADDRESS_ROWS = (
    heerlen_addresses_df[["full_address", "coordinates"]]
    .dropna(subset=["full_address"])
    .assign(full_address=lambda d: d["full_address"].astype(str).str.strip())
    .drop_duplicates(subset=["full_address"])
    .to_dict("records")
)

if not VALID_HEERLEN_ADDRESS_ROWS:
    raise ValueError("No usable rows found in heerlen_addresses_df['full_address'].")

random.shuffle(VALID_HEERLEN_ADDRESS_ROWS)
_ADDRESS_INDEX = 0


def generate_real_address(max_attempts: int = 1):
    """Return one known Heerlen address row with its coordinates."""
    global _ADDRESS_INDEX
    row = VALID_HEERLEN_ADDRESS_ROWS[_ADDRESS_INDEX % len(VALID_HEERLEN_ADDRESS_ROWS)]
    _ADDRESS_INDEX += 1
    return row


def generate_employee(index):
    """Generate a single employee record as a dictionary."""
    name = f"employee {index}"

    address_row = generate_real_address()
    address = address_row["full_address"]
    coordinates = address_row.get("coordinates")

    time_window_start = "07:00"
    time_window_end = "18:00"

    dogs = random.choices([0, 1, 2, -1], weights=[0.2, 0.2, 0.1, 0.5])[0]
    cats = random.choices([0, 1, 2, -1], weights=[0.1, 0.3, 0.1, 0.5])[0]
    smokes = random.random() < 0.3

    return {
        "name": name,
        "address": address,
        "coordinates": coordinates,
        "time_window_start": time_window_start,
        "time_window_end": time_window_end,
        "dogs": dogs,
        "cats": cats,
        "smokes": smokes,
    }


def generate_employees_csv(num_employees, filename="../output/employees.csv"):
    """Generate num_employees records and write them to a CSV file."""
    fieldnames = [
        "name", "address", "coordinates",
        "time_window_start", "time_window_end",
        "dogs", "cats", "smokes",
    ]

    output_path = Path(filename)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    with open(output_path, mode="w", newline="", encoding="utf-8") as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        for i in range(1, num_employees + 1):
            employee = generate_employee(i)
            employee["smokes"] = str(employee["smokes"]).lower()
            writer.writerow(employee)

    print(f"Generated {num_employees} employees in '{output_path}'.")


if __name__ == "__main__":
    generate_employees_csv(20)

Generated 20 employees in '..\output\employees.csv'.
